# 第 20 课：CTC + 语言模型——Prefix Beam、LM scale 与 Hotword

目标：在“真正扩展输出 token”时加入 LM 分数，并理解插入惩罚和热词偏置。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 语言模型与 WFST |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 19 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 浅融合、LM scale/插入项、hotword |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：浅融合、LM scale/插入项、hotword。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

from collections import defaultdict
from ipywidgets import interact, FloatSlider, IntSlider
labels=[BLANK,"天","田","气"]
P=np.array([[.55,.10,.50,.10,.55],[.22,.42,.15,.10,.10],[.18,.38,.15,.10,.10],[.05,.10,.20,.70,.25]])
bigram={("天","气"):.75,("田","气"):.18,("<s>","天"):.55,("<s>","田"):.35}
def lm_prob(prefix,c): return bigram.get(((prefix[-1] if prefix else "<s>"),c),.05)

## 1. 融合分数

常见形式：`acoustic + α × LM + β × token_count + hotword_bonus`。不同实现使用概率或负对数代价，符号方向必须统一。

In [ ]:
def decode(P,beam_size=8,alpha=.0,beta=.0,hotword="",hot_bonus=0.0):
    beam={"":(1.0,0.0,0.0)}  # pb,pnb,额外log分数
    for t in range(P.shape[1]):
        nxt=defaultdict(lambda:[0.0,0.0,-np.inf])
        for pref,(pb,pnb,extra) in beam.items():
            for i,c in enumerate(labels):
                p=P[i,t]
                if c==BLANK:
                    nxt[pref][0]+=(pb+pnb)*p; nxt[pref][2]=max(nxt[pref][2],extra)
                elif pref and c==pref[-1]:
                    nxt[pref][1]+=pnb*p; nxt[pref][2]=max(nxt[pref][2],extra)
                    new=pref+c; add=alpha*np.log(lm_prob(pref,c))+beta+(hot_bonus if hotword and new.endswith(hotword) else 0)
                    nxt[new][1]+=pb*p; nxt[new][2]=max(nxt[new][2],extra+add)
                else:
                    new=pref+c; add=alpha*np.log(lm_prob(pref,c))+beta+(hot_bonus if hotword and new.endswith(hotword) else 0)
                    nxt[new][1]+=(pb+pnb)*p; nxt[new][2]=max(nxt[new][2],extra+add)
        score=lambda item: np.log(sum(item[1][:2])+1e-30)+item[1][2]
        beam={k:tuple(v) for k,v in sorted(nxt.items(),key=score,reverse=True)[:beam_size]}
    score=lambda item: np.log(sum(item[1][:2])+1e-30)+item[1][2]
    return sorted(beam.items(),key=score,reverse=True)

for alpha in [0,.5,1.0]: print("alpha",alpha,"best",decode(P,alpha=alpha)[0][0])

## 2. 交互调 LM scale、插入项和热词

In [ ]:
@interact(alpha=FloatSlider(min=0,max=2,value=.5,step=.1),beta=FloatSlider(min=-1,max=1,value=0,step=.1),hot_bonus=FloatSlider(min=0,max=3,value=0,step=.25))
def tune(alpha=.5,beta=0,hot_bonus=0):
    result=decode(P,alpha=alpha,beta=beta,hotword="天气",hot_bonus=hot_bonus)
    for pref,state in result[:5]: print(pref,state)

## 3. Hotword 不是无条件替换

热词应该只在声学候选仍合理时加有限 bonus。过强会把不相关语音强行识别成热词。生产系统需要在领域召回率和误触发率之间调参。

## 本课测试

1. LM 分数为什么只在前缀真正扩展时加入？
2. LM scale=0 表示什么？
3. 正的 token insertion bonus 通常偏向长还是短输出？
4. 热词权重越大是否越好？
5. 为什么调参必须同时看 WER/CER 与延迟？

<details><summary>展开参考答案</summary>

1. blank 和不产生新 token 的重复不应重复计算 LM。2. 不使用 LM。3. 偏向更长输出。4. 不是，过大会误触发。5. 更复杂的搜索可能提高准确率但增加计算和延迟。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 20 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `浅融合`、`LM scale/插入项`、`hotword`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**热词 bonus 过大**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**扫描 alpha/beta/bonus 并画准确率—偏置曲线**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接 Prefix Beam 与 G 图**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：浅融合、LM scale/插入项、hotword。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 浅融合、LM scale/插入项、hotword。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
